# AI Agent Building — Runnable Companion Notebook

Runnable code for every pattern described in [`README.md`](./README.md). Two tracks:

1. **Part 1 — Mock LLM** (no API key, no internet, runs anywhere) — see the *think → act → observe → repeat* loop mechanics with full step-by-step tracing.
2. **Part 2 — Real Groq-powered agent** (needs `pip install groq` + a free key from console.groq.com) — the production version from README Section 4.
3. **Part 3 — Guardrails demo** — what happens without a max-step limit, and why you need one.

Run top to bottom. Part 2 auto-skips gracefully if `groq` isn't installed or `GROQ_API_KEY` isn't set — Part 1 and Part 3 always run.

---
## 0. Setup

In [1]:
import json
import os

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
try:
    from groq import Groq
    HAS_GROQ_SDK = True
except ImportError:
    HAS_GROQ_SDK = False

print("groq SDK installed:", HAS_GROQ_SDK)
print("GROQ_API_KEY set:", bool(GROQ_API_KEY))

groq SDK installed: False
GROQ_API_KEY set: False


---
## 1. Define tools (Step 2 & 3 from the README)

Same two tools used in both the mock and the real version, so you can compare the loop trace directly.

In [2]:
def get_weather(city: str) -> str:
    """Get the current weather for a city (fake data for demo purposes)."""
    fake_data = {"dhaka": "32°C, humid", "london": "15°C, rainy", "new york": "22°C, clear"}
    return fake_data.get(city.lower(), f"No weather data for '{city}'")


def calculate(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '15% of 3200' -> '15/100*3200'."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"


available_tools = {"get_weather": get_weather, "calculate": calculate}

tools_schema = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "parameters": {"type": "object",
                        "properties": {"city": {"type": "string"}},
                        "required": ["city"]}}},
    {"type": "function", "function": {
        "name": "calculate",
        "description": "Evaluate a basic math expression, e.g. \'12 * (3 + 4)\'.",
        "parameters": {"type": "object",
                        "properties": {"expression": {"type": "string"}},
                        "required": ["expression"]}}},
]

print("Tools registered:", list(available_tools.keys()))

Tools registered: ['get_weather', 'calculate']


---
## 2. Part 1 — The agent loop with a mock "LLM" (no API key needed)

This fakes what a real tool-calling LLM response looks like, so you can see **exactly** what the
orchestrator loop does at every step — independent of any API. This is the same loop shape as
Section 4/5 of the README, just with `call_mock_llm()` standing in for a real API call.

In [3]:
class MockToolCall:
    """Mimics the shape of a real SDK's tool_call object (Groq/OpenAI-style)."""
    def __init__(self, call_id, name, arguments: dict):
        self.id = call_id

        class _Fn:
            pass
        self.function = _Fn()
        self.function.name = name
        self.function.arguments = json.dumps(arguments)


class MockMessage:
    def __init__(self, content=None, tool_calls=None):
        self.content = content
        self.tool_calls = tool_calls or []


def call_mock_llm(messages, step_counter=[0]):
    """
    A deterministic stand-in for a real LLM: on the FIRST call it decides to use
    tools; once it sees tool results in the message history, it gives a final answer.
    A real LLM makes this decision by reasoning over the prompt -- here it's scripted
    so the loop mechanics are 100% visible and reproducible.
    """
    have_tool_results = any(m.get("role") == "tool" for m in messages)

    if not have_tool_results:
        # "decide" to call both tools based on the user question
        return MockMessage(tool_calls=[
            MockToolCall("call_1", "get_weather", {"city": "Dhaka"}),
            MockToolCall("call_2", "calculate", {"expression": "15/100*3200"}),
        ])
    else:
        weather = next(m["content"] for m in messages if m.get("name") == "get_weather")
        calc = next(m["content"] for m in messages if m.get("name") == "calculate")
        return MockMessage(content=f"The weather in Dhaka is {weather}. 15% of 3200 is {calc}.")

In [4]:
def run_agent_mock(user_message: str, max_steps: int = 5, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful assistant with access to tools."},
        {"role": "user", "content": user_message},
    ]

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"--- Step {step}: calling LLM ---")
        msg = call_mock_llm(messages)

        if not msg.tool_calls:
            if verbose:
                print("LLM gave a final answer, stopping.\n")
            return msg.content

        messages.append({"role": "assistant", "content": None,
                          "tool_calls": [{"id": c.id, "function": {"name": c.function.name,
                                                                    "arguments": c.function.arguments}}
                                         for c in msg.tool_calls]})
        for call in msg.tool_calls:
            fn = available_tools[call.function.name]
            args = json.loads(call.function.arguments)
            result = fn(**args)
            if verbose:
                print(f"  Action: {call.function.name}({args}) -> Observation: {result}")
            messages.append({"role": "tool", "tool_call_id": call.id,
                              "name": call.function.name, "content": str(result)})

    return "Stopped: too many steps without a final answer."


answer = run_agent_mock("What is the weather in Dhaka, and what is 15% of 3200?")
print("\nFinal answer:", answer)

--- Step 1: calling LLM ---
  Action: get_weather({'city': 'Dhaka'}) -> Observation: 32°C, humid
  Action: calculate({'expression': '15/100*3200'}) -> Observation: 480.0
--- Step 2: calling LLM ---
LLM gave a final answer, stopping.


Final answer: The weather in Dhaka is 32°C, humid. 15% of 3200 is 480.0.


---
## 3. Part 2 — Real Groq-powered agent (Section 4 of the README)

Needs `pip install groq` and a free API key from [console.groq.com](https://console.groq.com), set as
the `GROQ_API_KEY` environment variable before starting Jupyter. This cell auto-skips if either is missing,
so the notebook still runs end-to-end without them.

In [5]:
def run_agent_groq(user_message: str, max_steps: int = 5) -> str:
    client = Groq(api_key=GROQ_API_KEY)
    model = "llama-3.3-70b-versatile"

    messages = [
        {"role": "system", "content": "You are a helpful assistant with access to tools. "
                                       "Only call a tool when you need it. Otherwise answer directly."},
        {"role": "user", "content": user_message},
    ]

    for _ in range(max_steps):
        response = client.chat.completions.create(
            model=model, messages=messages, tools=tools_schema, tool_choice="auto",
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content

        messages.append(msg)
        for call in msg.tool_calls:
            fn = available_tools[call.function.name]
            args = json.loads(call.function.arguments)
            result = fn(**args)
            messages.append({"role": "tool", "tool_call_id": call.id,
                              "name": call.function.name, "content": str(result)})

    return "Stopped: too many steps without a final answer."


if HAS_GROQ_SDK and GROQ_API_KEY:
    print(run_agent_groq("What is the weather in Dhaka, and what is 15% of 3200?"))
else:
    missing = []
    if not HAS_GROQ_SDK:
        missing.append("`pip install groq`")
    if not GROQ_API_KEY:
        missing.append("`GROQ_API_KEY` environment variable")
    print("Skipping live call - missing:", " and ".join(missing))
    print("The mock version above (Part 1) demonstrates the identical loop logic.")

Skipping live call - missing: `pip install groq` and `GROQ_API_KEY` environment variable
The mock version above (Part 1) demonstrates the identical loop logic.


---
## 4. Part 3 — Guardrails demo

What happens **without** a max-step limit: a "confused" mock LLM that keeps calling a tool and never
gives a final answer. This is exactly the failure mode `max_steps` (Section 6/8 of the README) protects
against in production.

In [6]:
def call_confused_mock_llm(messages):
    """Always calls a tool, never gives a final answer -- simulates a stuck/looping agent."""
    return MockMessage(tool_calls=[MockToolCall(f"call_{len(messages)}", "get_weather", {"city": "Dhaka"})])


def run_agent_confused(user_message: str, max_steps: int = 5) -> str:
    messages = [{"role": "user", "content": user_message}]
    for step in range(1, max_steps + 1):
        msg = call_confused_mock_llm(messages)
        if not msg.tool_calls:
            return msg.content
        for call in msg.tool_calls:
            result = available_tools[call.function.name](**json.loads(call.function.arguments))
            messages.append({"role": "tool", "name": call.function.name, "content": result})
            print(f"Step {step}: called {call.function.name} again -> {result}")
    return "Stopped by max_steps guardrail -- without this limit, this would loop forever."


print(run_agent_confused("What is the weather?", max_steps=4))

Step 1: called get_weather again -> 32°C, humid
Step 2: called get_weather again -> 32°C, humid
Step 3: called get_weather again -> 32°C, humid
Step 4: called get_weather again -> 32°C, humid
Stopped by max_steps guardrail -- without this limit, this would loop forever.


---
## 5. Recap

| Part | Shows |
|---|---|
| 1 — Mock LLM | The think -> act -> observe -> repeat loop, fully traced, no API needed |
| 2 — Real Groq agent | The same loop driven by an actual tool-calling LLM |
| 3 — Guardrails | Why `max_steps` exists -- without it, a stuck agent loops forever |

Full walkthrough and the theory behind each step: see [`README.md`](./README.md).